# Ансамблевые методы в машинном обучении

### Семинар: от одного дерева к лесу и бустингу

In [ ]:
# pip3 install numpy pandas matplotlib seaborn scikit-learn catboost xgboost lightgbm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import (
    BaggingRegressor, RandomForestRegressor,
    StackingRegressor, VotingRegressor
)
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Датасет

В сегодняшнем семинаре воспользуемся датасетом [Student Exam Performance Dataset Analysis](https://www.kaggle.com/datasets/grandmaster07/student-exam-performance-dataset-analysis).

В нем нам необходимо предсказать оценку за экзамен, при этом в наличии у нас 19 фич: от физической активности до посещаемости.

In [ ]:
df = pd.read_csv('/home/darinka/course_ML_HSE/lectures/5_ensembles/StudentPerformanceFactors.csv')

In [ ]:
df.head()

In [ ]:
df.info()

Всего 6607 значений, подавляющее большинство не имеет пропущенных значений. Что хорошо для нас!

Также можно отметить много категориальных фич

### Подготовка данных

Давайте разделим категориальные фичи от числовых - это нам пригодится для кодирования

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

target_col = 'Exam_Score' 
num_cols = [c for c in num_cols if c != target_col]

In [ ]:
print(f'Категориальные признаки ({len(cat_cols)}): {cat_cols}')
print(f'Числовые признаки ({len(num_cols)}): {num_cols}')

Так как у нас нет пропущенных значений в численных колонках, заполним категориальные колонки 'missing'

---

**Вопрос аудитории**: почему бы просто не удалить такие значения?

---

In [ ]:
df[cat_cols] = df[cat_cols].fillna('missing')


Разделим на трейн и тест сет:

In [ ]:
features = [c for c in df.columns if c != target_col]

X = df[features]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}')
print(f'Признаков: {X_train.shape[1]}')

В этом занятии не будем делать EDA, сфокусируемся на изучении ансамблей.

## Вспоминаем Decision Tree

**Дерево решений** - это алгоритм, который рекурсивно разбивает пространство признаков по придкату на прямоугольные области и в каждой области делает константное предсказание.

На каждом шаге дерево перебирает пары (признак $j$, порог $t$) и оценивает, насколько разбиение улучшает качество $Q(X, j, t)$:

$$
Q(X, j, t) = H(X_m) - \frac{H(X_l)}{|X_l|} - \frac{H(X_r)}{|X_r|}
$$

где $H(X)$ - это **информативность** выборки. Чем она ниже, тем лучше - в выборке много похожих элементов.

При этом информативность может определяться по-разному в зависимости от задачи: для классификации обычно используют Gini/entropy, а в регрессии - MSE.


Для нашей задачи будем использовать MSE. А метрику качества - R2.

![Визуализация дерева решений](image.png)

Создадим функцию, которая будет фиксировать для текущего алгоритма значения метрик, чтобы в конце их сравнить.

---

**Вопрос аудитории**: что оценивает метрика R2?

---

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name='Model'):
    """Считает метрики и возвращает словарь."""
    y_pred_tr = model.predict(X_tr)
    y_pred_te = model.predict(X_te)
    results = {
        'Model': name,
        'RMSE_train': np.sqrt(mean_squared_error(y_tr, y_pred_tr)),
        'RMSE_test': np.sqrt(mean_squared_error(y_te, y_pred_te)),
        'R2_test': r2_score(y_te, y_pred_te),
    }
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  RMSE  train: {results['RMSE_train']:>10,.1f}  |  test: {results['RMSE_test']:>10,.1f}")
    print(f"  R²    test:  {results['R2_test']:>10.4f}")
    return results


all_results = []

Давайте обработаем категориальны фичи методом `TargetEncoder` - его мы обсуждали на предыдущей лекции.

---

**Вопрос аудитории**: что делает метод `TargetEncoder`?

---

При этом в реализации от `sklearn` фичи кодируются по фолдам, чтобы избежать даталика. 

Воспльзуемся `ColumnTransformer` чтобы применить `TargetEncoder` только к категориальным фичам.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder

feature_prep = ColumnTransformer(
    transformers=[
        (
            "cat",
            TargetEncoder(
                target_type="continuous",
                cv=5,
                random_state=RANDOM_STATE,
            ),
            cat_cols,
        ),
    ],
    remainder="passthrough",
)

X_train_enc = feature_prep.fit_transform(X_train, y_train)
X_test_enc = feature_prep.transform(X_test)

In [ ]:
X_train_enc[0]

Обучим наше первое дерево. Для этого воспользуемся [DecisionTreeRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html). Для классификации использовали бы `DecisionTreeClassifier`.

Для начала не будем настраивать гиперпараметры - возьмем те, что по дефолту.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
tree.fit(X_train_enc, y_train)

res = evaluate(tree, X_train_enc, y_train, X_test_enc, y_test, 'Decision Tree (no limit)')
all_results.append(res)

print(f'Глубина дерева: {tree.get_depth()}')
print(f'Количество листьев: {tree.get_n_leaves()}')

---

**Вопрос аудитории**: о чем нам говорит `RMSE train = 0.0`. Что еще можно отметить в обученном дереве, какое оно у нас получилось? Что мы можем сделать, чтобы избавиться от <>? Вспоминаем про критерий останова из предыдущей лекции

---

In [ ]:
tree = DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=10)
tree.fit(X_train_enc, y_train)

res = evaluate(tree, X_train_enc, y_train, X_test_enc, y_test, 'Decision Tree (max_depth=10)')
all_results.append(res)

print(f'Глубина дерева: {tree.get_depth()}')
print(f'Количество листьев: {tree.get_n_leaves()}')

Уже лучше - дерево получилось не таким переобученным. Благодаря этому метрика качества выросла.

In [ ]:
tree = DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=10, min_samples_leaf=100)
tree.fit(X_train_enc, y_train)

res = evaluate(tree, X_train_enc, y_train, X_test_enc, y_test, 'Decision Tree (max_depth=10, min_samples_leaf=100)')
all_results.append(res)

print(f'Глубина дерева: {tree.get_depth()}')
print(f'Количество листьев: {tree.get_n_leaves()}')

In [ ]:
tree = DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=2)
tree.fit(X_train_enc, y_train)

res = evaluate(tree, X_train_enc, y_train, X_test_enc, y_test, 'Decision Tree (max_depth=2)')
all_results.append(res)

print(f'Глубина дерева: {tree.get_depth()}')
print(f'Количество листьев: {tree.get_n_leaves()}')

In [ ]:
pd.DataFrame(all_results).sort_values("R2_test", ascending=False)

Давайте сгенерируем данные чтобы визуализировать переобучение

Возьмем рандомные числа и будем применять функцию `true_function` - в данном случае это `x * sin(x)`. Дополнительно добавим к тарагетам немного шума из нормального распределения.

In [ ]:
np.random.seed(RANDOM_STATE)

def true_function(x):
    return x * np.sin(x)

def generate_data(n_samples: int = 50, noise: float = 3, n_noise_samples: int = 1):
    # генерируем случайные входные значения x в диапазоне [-10, 10] и сортируем их
    x = np.random.rand(n_samples) * 20 - 10
    x = np.sort(x)

    y = np.zeros((n_samples, n_noise_samples))
    # для каждого прогона добавляем шум к истинной функции
    for i in range(n_noise_samples):
        y[:, i] = true_function(x) + np.random.normal(0.0, noise, n_samples)

    return x.reshape((n_samples, 1)), y.squeeze()

Сгенерируем с помощью этой функции 3 разных датасета (но при этом они будут иметь одну и ту же целевую функцию).

На каждом датасете обучим 3 дерева разной глубины: 1, 4, 20

In [ ]:
x_plot = np.linspace(-10, 10, 300).reshape(-1, 1)
y_true = true_function(x_plot.ravel())

depths = [1, 4, 20]
depth_labels = ['depth=1 (Underfitting)', 'depth=4 (В самый раз)', 'depth=20 (Overfitting)']
colors = ['#2196F3', '#4CAF50', '#F44336']
n_datasets = 3

fig, axes = plt.subplots(n_datasets, 3, figsize=(18, 5 * n_datasets))

datasets = [generate_data(noise=3) for _ in range(n_datasets)]

for row, (X_demo, y_demo) in enumerate(datasets):
    for col, (depth, title, color) in enumerate(zip(depths, depth_labels, colors)):
        ax = axes[row, col]
        tree = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE)
        tree.fit(X_demo, y_demo)
        y_pred = tree.predict(x_plot)

        ax.scatter(X_demo, y_demo, alpha=0.5, s=25, color='gray', label='Данные')
        ax.plot(x_plot, y_true, 'k--', alpha=0.5, linewidth=1.5, label='Истинная f(x)')
        ax.plot(x_plot, y_pred, color=color, linewidth=2.5, label=f'Дерево (depth={depth})')
        ax.set_title(f'Выборка {row + 1} | {title}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)

plt.suptitle('Деревья решений с разной глубиной на 3 разных подвыборках', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

**Вопрос аудитории**: деревья какой глубины больше всего похожи друг на друга? а какие меньше всего похожи?

---

Давайте попробуем на одном графике визуализировать 50 деревьев обученных на разных подвыборках размера 100 (одной и той же целевой функции!) разной глубины и посмотрим, что между ними общего.

In [ ]:
N_TREES = 50
n_samples = 100

def train_classifier_on_random_subset(clf_class, parameters, n_train_samples: int = 20, noise: float = 3):
    """Обучает один классификатор на случайной подвыборке."""
    x_train, y_train = generate_data(n_samples=n_train_samples, noise=noise)
    clf = clf_class(**parameters)
    clf.fit(x_train, y_train)
    return clf, x_train, y_train


predictions_d4 = []
predictions_d20 = []

for i in range(N_TREES):
    X_i, y_i = generate_data(100)

    tree4 = DecisionTreeRegressor(max_depth=4, random_state=i)
    tree4.fit(X_i, y_i)
    predictions_d4.append(tree4.predict(x_plot))

    tree20 = DecisionTreeRegressor(max_depth=20, random_state=i)
    tree20.fit(X_i, y_i)
    predictions_d20.append(tree20.predict(x_plot))

predictions_d4 = np.array(predictions_d4)
predictions_d20 = np.array(predictions_d20)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, preds, depth, color in zip(axes, [predictions_d4, predictions_d20],
                                    [4, 20], ['#4CAF50', '#F44336']):
    for i in range(N_TREES):
        ax.plot(x_plot, preds[i], alpha=0.08, color=color)

    mean_pred = preds.mean(axis=0)
    std_pred = preds.std(axis=0)

    ax.plot(x_plot, y_true, 'k--', linewidth=2, label='Истинная f(x)')
    ax.plot(x_plot, mean_pred, color=color, linewidth=3, label=f'Среднее {N_TREES} деревьев')
    # ax.fill_between(x_plot.ravel(), mean_pred - 2*std_pred, mean_pred + 2*std_pred,
    #                 alpha=0.2, color=color, label='± 2σ (разброс)')
    ax.set_title(f'depth={depth} — {N_TREES} деревьев на разных подвыборках', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)

plt.suptitle('Variance деревьев: каждое дерево — бледная линия, среднее — жирная',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

**Вопрос аудитории**: что вы можете сказать про деревья разной глубины? Какое "среднее" от выходов всех деревьев ближе к таргету? Деревья какой глубины наиболее разнообразны? как это можно объяснить более формально?

---

In [ ]:
mean_pred_d4 = predictions_d4.mean(axis=0)
mean_pred_d20 = predictions_d20.mean(axis=0)
bias2_d4 = ((mean_pred_d4 - y_true) ** 2).mean()
bias2_d20 = ((mean_pred_d20 - y_true) ** 2).mean()

print(f'variance (depth=4):  {predictions_d4.var(axis=0).mean():.4f}')
print(f'variance (depth=20): {predictions_d20.var(axis=0).mean():.4f}')
print(f'bias^2   (depth=4):  {bias2_d4:.4f}')
print(f'bias^2   (depth=20): {bias2_d20:.4f}')

Ясно, что bias^2 в 2 раза больше у неглубокого дерева, то есть среднее неглубоких деревьев меньше подстроилось именно под саму целевую функцию. А вот среднее глубоких деревьев почти идеально предсказало целевую функцию - однако дисперсия у таких деревьев выше.

## Bias-Variance Tradeoff

### Разложение ошибки

Для любой модели $\hat{f}(x)$ ожидаемая ошибка предсказания раскладывается на три слагаемых:

$$\mathbb{E}[\mathcal{L}(y, \hat{f}(x))] = \text{Bias}^2[\hat{f}(x)] + \text{Var}[\hat{f}(x)] + \sigma^2$$

[Здесь](https://education.yandex.ru/handbook/ml/article/bias-variance-decomposition) можно подробнее изучить как мы получаем такое разложение.

### Что значит каждое слагаемое?

**Bias (смещение)** — насколько в среднем предсказания модели отличаются от истинной функции.

$$
\text{Bias}[\hat{f}(x)] = \mathbb{E}[\hat{f}(x)] - f(x)
$$

- Высокий bias → модель слишком простая, **недообучение** (underfitting)
- Например, лучник, который целится мимо мишени. Не важно сколько раз стреляет — всё время попадает в одно неправильное место

**Variance (дисперсия)** — насколько сильно меняются предсказания модели при обучении на разных подвыборках.

$$
\text{Var}[\hat{f}(x)] = \mathbb{E}\left[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\right]
$$

- Высокая variance → модель слишком сложная, **переобучение** (overfitting)
- Например, лучник целится правильно, но рука дрожит — стрелы разлетаются по всей мишени

**Шум ($\sigma^2$)** — неустранимый шум в данных. Сколько бы мы ни улучшали модель, эту часть ошибки убрать невозможно.

![](tradeoff.png)

![](лучник.png)

Давайте посмотрим чуть подробнее, как меняется bias и variance у ошибки при разной глубине дерева.

Для каждой глубины (от 2 до 15) будем 1000 раз генерировать данные из целевой функции - все той же, с определенным шумом (в данном случае 3). Размер выборки будет 500

Далее разложим ошибку каждой глубины на составляющие - bias, variance и шум

In [ ]:
N_TRAIN_SETS   = 1000   # количество случайных обучающих подмножеств
N_TRAIN        = 500    # размер каждого обучающего подмножества
N_TEST         = 500    # размер тестового множества точек x_test
NOISE_STD      = 3.0   # стандартное отклонение шума
DEPTHS         = list(range(2, 16))

np.random.seed(RANDOM_STATE)

x_test = np.sort(np.random.uniform(-10, 10, N_TEST))

f_test = true_function(x_test)
noise_val = NOISE_STD ** 2

results = []

for depth in DEPTHS:
    preds = np.zeros((N_TRAIN_SETS, N_TEST))
    for i in range(N_TRAIN_SETS):
        X_i, y_i = generate_data(n_samples=N_TRAIN, noise=NOISE_STD)
        tree = DecisionTreeRegressor(max_depth=depth, random_state=i)
        tree.fit(X_i, y_i)
        preds[i] = tree.predict(x_test.reshape(-1, 1))

    mean_pred = preds.mean(axis=0)                          # E_D[a(x)]
    bias2     = ((mean_pred - f_test) ** 2).mean()          # E_x[(E_D[a]-f)²]
    variance  = ((preds - mean_pred) ** 2).mean()           # E_x,D[(a - E_D[a])²]
    mse       = bias2 + variance + noise_val                # разложение ошибки

    results.append({
        'Глубина': depth,
        'MSE': round(mse, 3),
        'Bias^2': round(bias2, 3),
        'Variance': round(variance, 3),
        'Noise': round(noise_val, 3),
    })

df_decomp = pd.DataFrame(results).set_index('Глубина')

In [ ]:
df_decomp

In [ ]:
def plot_bias_variance(bias_variance_results: pd.DataFrame, 
                       target_parameter_name: str,  target_parameter_values) -> None:
  plt.figure(figsize=(8, 5), dpi=150)
  plt.xticks(target_parameter_values)
  plt.plot(target_parameter_values, bias_variance_results["Bias^2"], label="Bias^2", color="blue")
  plt.plot(target_parameter_values, bias_variance_results["Variance"], label="Variance", color="orange")
  plt.plot(target_parameter_values, bias_variance_results["Noise"], label="Noise", color="green")
  plt.plot(target_parameter_values, bias_variance_results["MSE"], label="MSE", color="red")
  plt.xlabel(target_parameter_name)
  plt.legend(fontsize=10, loc="upper right")
  plt.show()

In [ ]:
depth = list(DEPTHS)

plot_bias_variance(df_decomp, target_parameter_name="Tree depth",  target_parameter_values=depth)

### Выводы про деревья

Раз мы знаем, что ошибка раскладывается на bias, variance и шум, будет логично, что мы хотим уменьшить хотя бы одну из составляющих ошибки.

Шум мы никак уменьшить не можем - это минимальная ошибка.

Давайте попробуем уменьшить variance. Мы знаем, что у глубоких деревьев низкий bias, но высокий variance. Что если уменьшить именно вторую составляющую?

Пример приходит из жизни древних греков: если много людей проголосуют независимо друг от друга, то вместе они придут к разумному решению, несмотря на то что опыт каждого из них субъективен. Аналог голосования в мире машинного обучения — бэггинг. кусок [отсюда](https://education.yandex.ru/handbook/ml/article/ansambli-v-mashinnom-obuchenii)

## Bagging и Random Forest

### Идея Bagging (Bootstrap AGGregatING)

Помните метафору с лучником? Bagging - это много лучников, каждый тренировался на чуть разных мишенях (bootstrap-выборках), а мы берём среднее их выстрелов.

**Алгоритм:**
1. Из обучающей выборки размера $N$ делаем $K$ bootstrap-выборок (это выборки с возвращением) также размера $N$. Каждую выборку пометим как $X^i$
2. На каждой обучаем модель $b_i(x)$ на выборке $X^i$. В итоге получаем $K$ моделей.
3. Финальное предсказание строим как среднее всех $k$-предсказаний:

$$
a(x) = \frac{1}{K}\sum_{i=1}^{K} b_i(x)
$$

### Что происходит с bias?

Bias бэггинга равен bias одной модели. Докажем это:

$$
\text{Bias}[a(x)]
= \mathbb{E}[a(x)] - f(x)
= \mathbb{E}\!\left[\frac{1}{K}\sum_{i=1}^{K} b_i(x)\right] - f(x)
= \frac{1}{K}\sum_{i=1}^{K} \mathbb{E}[b_i(x)] - f(x)
$$

Так как все модели $b_i$ обучены по одному алгоритму на выборках того же размера, их распределения совпадают:
$\mathbb{E}[b_i(x)] = \mathbb{E}[b(x)]$. Поэтому:

$$
= \frac{1}{K} \cdot K \cdot \mathbb{E}[b(x)] - f(x) = \mathbb{E}[b(x)] - f(x) = \text{Bias}[b(x)]
$$

Бэггинг **не меняет bias**.

### Что происходит с variance?

Variance бэггинга меньше variance одной модели. Докажем это. Доказательство взято [отсюда](https://education.yandex.ru/handbook/ml/article/ansambli-v-mashinnom-obuchenii).

$$
\mathbb{V}[a(x, X)] = \mathbb{E}_X\!\left[a(x,X) - \mathbb{E}_X[a(x,X)]\right]^2 =
$$

*подставляем $a(x,X) = \dfrac{1}{k}\displaystyle\sum_{i=1}^{k} b(x, X_i)$*

$$
= \mathbb{E}_X\!\left[\frac{1}{k}\sum_{i=1}^{k} b(x,X_i) - \mathbb{E}_X\!\left[\frac{1}{k}\sum_{i=1}^{k} b(x,X_i)\right]\right]^2 =
$$

*выносим $\dfrac{1}{k}$ за скобки*

$$
= \frac{1}{k^2}\,\mathbb{E}_X\!\left[\sum_{i=1}^{k} b(x,X_i) - \mathbb{E}_X\!\left[\sum_{i=1}^{k} b(x,X_i)\right]\right]^2 =
$$

*вносим внутреннее $\mathbb{E}_X$ под знак суммы и объединяем два суммирования в одно*

$$
= \frac{1}{k^2}\,\mathbb{E}_X\!\left[\sum_{i=1}^{k}\Bigl(b(x,X_i) - \mathbb{E}_X b(x,X_i)\Bigr)\right]^2 =
$$

*раскрываем квадрат суммы: $\left(\sum_i z_i\right)^2 = \sum_i z_i^2 + \sum_{k_1 \neq k_2} z_{k_1} z_{k_2}$*

$$
= \frac{1}{k^2}\,\mathbb{E}_X\!\left[\sum_{i=1}^{k}\Bigl(b(x,X_i)-\mathbb{E}_X b(x,X_i)\Bigr)^2 + \sum_{k_1 \neq k_2}\Bigl(b(x,X_{k_1})-\mathbb{E}_X b(x,X_{k_1})\Bigr)\Bigl(b(x,X_{k_2})-\mathbb{E}_X b(x,X_{k_2})\Bigr)\right] =
$$

*раскрываем матожидание*

$$
= \frac{1}{k^2}\mathbb{E}_X\sum_{i=1}^{k}\Bigl(b(x,X_i)-\mathbb{E}_X b(x,X_i)\Bigr)^2 + \frac{1}{k^2}\mathbb{E}_X\sum_{k_1 \neq k_2}\Bigl(b(x,X_{k_1})-\mathbb{E}_X b(x,X_{k_1})\Bigr)\Bigl(b(x,X_{k_2})-\mathbb{E}_X b(x,X_{k_2})\Bigr) =
$$

*определения дисперсии и ковариации:*

$\mathbb{V}_X b(x,X_i) = \mathbb{E}_X\!\left[(b(x,X_i) - \mathbb{E}_X b(x,X_i))^2\right]$,

$\text{cov}(b(x,X_{k_1}), b(x,X_{k_2})) = \mathbb{E}_X\!\left[(b(x,X_{k_1}) - \mathbb{E}_X b(x,X_{k_1}))(b(x,X_{k_2}) - \mathbb{E}_X b(x,X_{k_2}))\right]$

$$
= \frac{1}{k^2}\sum_{i=1}^{k}\mathbb{V}_X b(x,X_i) + \frac{1}{k^2}\sum_{k_1 \neq k_2}\text{cov}\!\left(b(x,X_{k_1}),\, b(x,X_{k_2})\right)
$$

**Ковариация** $\text{cov}(b(x,X_{k_1}), b(x,X_{k_2}))$ — мера того, насколько два алгоритма «движутся вместе»: если одно дерево завышает предсказание на объекте $x$, завышает ли его и другое? Если модели не зависят друг от друга, ковариация равна нулю.

Если предположить, что базовые алгоритмы $b(x, X_i)$ **не коррелированы**, то есть $\text{cov}(b(x,X_{k_1}), b(x,X_{k_2})) = 0$ для любых $k_1 \neq k_2$, то:

$$
\mathbb{V}[a(x,X)] = \frac{1}{k^2}\sum_{i=1}^{k}\mathbb{V}_X b(x,X_i)
$$

Поскольку все выборки $X_i$ семплируются из одного и того же распределения $X$, индекс выборки не важен и $\mathbb{V}_X b(x, X_i) = \mathbb{V}_X b(x, X)$. Поэтому:

$$
\mathbb{V}[a(x,X)] = \frac{1}{k^2}\sum_{i=1}^{k}\mathbb{V}_X b(x,X_i) = \frac{1}{k^2}\sum_{i=1}^{k}\mathbb{V}_X b(x,X) = \frac{1}{k}\,\mathbb{V}_X b(x,X)
$$

Дисперсия композиции в $k$ раз меньше дисперсии отдельного алгоритма. 

![](bagging.png)

[сурс](https://aiml.com/what-is-bagging/)

Давайте повторим эксперимент с разной глубиной дерева и несколькими итерациями обучения. Но теперь будем сравнивать 2 алгоритма: 50 одиночных деревьев и 50 бэггингов (каждый бэггинг будет иметь 10 деревьев)

In [ ]:
# N — количество повторений эксперимента (разных выборок/моделей)
N = 500
# MAX_DEPTH — максимальная глубина дерева решений
MAX_DEPTH = 20
# N_TREES — число деревьев в ансамбле (к примеру, в бэггинге)
N_TREES = 10

In [ ]:
single_preds = []
bagging_preds = []

for i in range(N):
    X_i, y_i = generate_data(100)

    tree = DecisionTreeRegressor(max_depth=MAX_DEPTH, random_state=i)
    tree.fit(X_i, y_i)
    single_preds.append(tree.predict(x_plot))

    X_i, y_i = generate_data(10000)
    bag_i = BaggingRegressor(
        estimator=DecisionTreeRegressor(max_depth=MAX_DEPTH),
        n_estimators=N_TREES, random_state=i, max_samples=100
    )
    bag_i.fit(X_i, y_i)
    bagging_preds.append(bag_i.predict(x_plot))

single_preds = np.array(single_preds)
bagging_preds = np.array(bagging_preds)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, preds, label, color in zip(
    axes,
    [single_preds, bagging_preds],
    [f'Одно дерево (depth={MAX_DEPTH})', f'Бэггинг ({N_TREES} деревьев, depth={MAX_DEPTH})'],
    ['#F44336', '#4CAF50']
):
    for i in range(N):
        ax.plot(x_plot, preds[i], alpha=0.01, color=color)

    mean_pred = preds.mean(axis=0)
    std_pred = preds.std(axis=0)

    ax.plot(x_plot, y_true, 'k--', linewidth=2, label='Истинная f(x)')
    ax.plot(x_plot, mean_pred, color=color, linewidth=3, label=f'Среднее {N} моделей')
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)

plt.suptitle(f'Одно дерево vs Бэггинг: {N} повторений на разных выборках',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

mean_single = single_preds.mean(axis=0)
mean_bagging = bagging_preds.mean(axis=0)
bias2_single = ((mean_single - y_true) ** 2).mean()
bias2_bagging = ((mean_bagging - y_true) ** 2).mean()

print(f'Одно дерево  — variance: {single_preds.var(axis=0).mean():.4f},  bias²: {bias2_single:.4f}')
print(f'Бэггинг      — variance: {bagging_preds.var(axis=0).mean():.4f},  bias²: {bias2_bagging:.4f}')

Видим, что vairance упала почти в 10 раз - это количество деревьев в бэггинге!

Bias тоже упал. Но если попробуем повторить эксперимент с N = 1000, то bias почти не изменится.

Давайте теперь посмотрим чуть подробнее как на разных глубинах себя ведет бэггинг и как его ошибка раскладывается на bias-variance

In [ ]:
N_TRAIN_SETS   = 100     # количество случайных обучающих подмножеств
N_TRAIN        = 10000    # размер каждого обучающего подмножества
N_TEST         = 500      # размер тестового множества точек x_test
MAX_SAMPLES    = 500
NOISE_STD      = 3.0      # стандартное отклонение шума
N_TREES        = 10
DEPTHS         = list(range(2, 16))

np.random.seed(RANDOM_STATE)

x_test = np.sort(np.random.uniform(-10, 10, N_TEST))

f_test = true_function(x_test)
noise_val = NOISE_STD ** 2

results = []

for depth in DEPTHS:
    preds = np.zeros((N_TRAIN_SETS, N_TEST))
    for i in range(N_TRAIN_SETS):
        X_i, y_i = generate_data(n_samples=N_TRAIN, noise=NOISE_STD)
        tree = BaggingRegressor(
            estimator=DecisionTreeRegressor(max_depth=depth),
            n_estimators=N_TREES, random_state=i, max_samples=MAX_SAMPLES
        )
        tree.fit(X_i, y_i)
        preds[i] = tree.predict(x_test.reshape(-1, 1))

    mean_pred = preds.mean(axis=0)                          # E_D[a(x)]
    bias2     = ((mean_pred - f_test) ** 2).mean()          # E_x[(E_D[a]-f)²]
    variance  = ((preds - mean_pred) ** 2).mean()           # E_x,D[(a - E_D[a])²]
    mse       = bias2 + variance + noise_val                # разложение ошибки

    results.append({
        'Глубина': depth,
        'MSE': round(mse, 3),
        'Bias^2': round(bias2, 3),
        'Variance': round(variance, 3),
        'Noise': round(noise_val, 3),
    })

df_bagging = pd.DataFrame(results).set_index('Глубина')

In [ ]:
df_bagging

In [ ]:
depth = list(DEPTHS)

plot_bias_variance(df_bagging, target_parameter_name="Tree depth",  target_parameter_values=depth)

In [ ]:
plot_bias_variance(df_decomp, target_parameter_name="Tree depth",  target_parameter_values=depth)

Variance у бэггинга также растет с ростом глубины - однако в сравнении с одним деревом понятно, что общая ошибка падает.

Обучим наконец бэггинг на наших данных и посмотрим результат

In [ ]:
tree = BaggingRegressor(
            estimator=DecisionTreeRegressor(max_depth=20),
            n_estimators=100, random_state=RANDOM_STATE
)
tree.fit(X_train_enc, y_train)

res = evaluate(tree, X_train_enc, y_train, X_test_enc, y_test, 'Bagging (10 trees, max_depth=10)')
all_results.append(res)

In [ ]:
pd.DataFrame(all_results).sort_values("R2_test", ascending=False)

Бэггинг дал силььное улучшение по сравнению с единичным деревом. И это мы еще не тюнили гиперпараметры!

### Random Forest = Bagging + случайные подмножества фич

До этого при доказательстве мы сделали предположение, что модели не скоррелированы. Но это может быть не совсем правдой - мы учим модели на одну и ту же зависимость, на выборках, которые пересекаются.

Поэтому нам важно, чтобы алгоритмы не были похожи друг на друга. Как этого можно добиться?

Random Forest: 

1) Также будем бутстрапить выборку для каждого алгоритма $b_i(x)$
2) Но теперь при построении каждой модели в каждой вершине случайно выбирать $k < D$ признаков, где $D$ - количество признаков. И будем уже из них выбирать оптимальный сплит.
3) Также как и в обычном бэггинге усредняем или берем самый популярный класс 

Это снижает корреляцию между деревьями, а значит и variance.

In [ ]:
tree = RandomForestRegressor(
    n_estimators=100,         # количество деревьев в ансамбле
    max_features='sqrt',      # количество признаков для поиска лучшего разбиения в каждой вершине
    max_depth=20,             # максимальная глубина каждого дерева
    random_state=RANDOM_STATE,
    oob_score=True            # включает вычисление OOB-оценки качества (out-of-bag score)
)

tree.fit(X_train_enc, y_train)

res = evaluate(tree, X_train_enc, y_train, X_test_enc, y_test, 'Random Forest (100 trees, max_depth=20)')
all_results.append(res)

In [ ]:
pd.DataFrame(all_results).sort_values("R2_test", ascending=False)

Это не помогло нам улучшить метрику без тюна гиперпараметров...(

### OOB Score (Out-of-Bag Score) в Random Forest

#### Что такое OOB?

При построении случайного леса каждое дерево обучается на bootstrap-выборке — случайной подвыборке с возвращением из обучающего датасета. В среднем каждая такая выборка содержит примерно 63.2% уникальных объектов из оригинального датасета.
Оставшиеся ~36.8% объектов, которые не попали в bootstrap-выборку для данного дерева, называются Out-of-Bag (OOB) объектами для этого дерева.

#### Как считается OOB Score?
Для каждого объекта из обучающей выборки:

1) Находим все деревья, которые не использовали этот объект при обучении (это и есть его OOB-деревья)
2) Делаем предсказание для этого объекта только с помощью этих деревьев (агрегируем голоса/усредняем)
3) Сравниваем предсказание с истинным значением

OOB Score — это итоговая метрика качества (accuracy для классификации, R² для регрессии), посчитанная по всем таким OOB-предсказаниям.

#### Зачем это нужно?
OOB Score — это бесплатная оценка качества модели, которая:

- Не требует отдельной валидационной выборки
- Является несмещённой оценкой обобщающей способности (аналог кросс-валидации)
- Позволяет подбирать гиперпараметры без разделения данных на train/val


In [ ]:
tree.oob_score_

### Практика

Попробуйте изменить параметры Random Forest и посмотреть, как меняется качество и oob-score:
- `n_estimators`: 100, 200, 500 -> или любые другие
- `max_features`: `'sqrt'`, `'log2'`, `0.5`, `1.0`
- `max_depth`: `None`, `10`, `20`

Можете попробовать перебрать другие параметры, если почитаете документацию. **Дано 5-7 минут**

Используйте `GridSearchCV`. ИИ-инструменты тоже можно

In [ ]:
pd.DataFrame(all_results).sort_values("R2_test", ascending=False)

### Про гиперпараметры Random Forest

#### Глубина деревьев в случайном лесе
Используют глубокие деревья. Бэггинг уменьшает разброс, но не уменьшает смещение, поэтому важно, чтобы сами базовые модели имели низкое смещение. Неглубокие деревья дают простые и устойчивые, но неточные предсказания (высокое смещение). Глубокие деревья хорошо подстраиваются под данные (низкое смещение), но переобучаются - это компенсируется ансамблированием. Итог: лучше брать глубокие деревья.

#### Сколько признаков использовать
Количество признаков регулирует корреляцию между деревьями и их силу. Если брать много признаков, деревья становятся похожими друг на друга, и ансамбль работает хуже. Если брать слишком мало признаков, деревья становятся слабыми. Нужно балансировать: уменьшать корреляцию, но не делать деревья слишком слабыми. Обычно используют корень из количества признаков.

#### Сколько деревьев в случайном лесе
Увеличение числа деревьев уменьшает разброс, но не влияет на смещение. Улучшение качества со временем замедляется, потому что разнообразие деревьев ограничено. Обычно подбирают число деревьев по графику ошибки: увеличивают, пока качество заметно растёт. Ограничение также накладывает время работы - больше деревьев работают дольше, хотя обучение и предсказание можно распараллелить.

### Выводы про Bagging / Random Forest

Бэггинг: обучаем много моделей на bootstrap-подвыборках и усредняем предсказания. Смещение остаётся примерно как у одной модели, зато разброс сильно падает - поэтому базовые алгоритмы берут достаточно сложными (глубокие деревья), а не «слишком простыми».

Случайный лес - тот же бэггинг над деревьями, плюс на каждом сплите случайное подмножество признаков: деревья менее коррелированы, ансамбль разнообразнее. Число деревьев и сила признакового сэмплинга балансируют между корреляцией деревьев и их «силой».

На практике RF часто даёт хорошее качество без тонкой настройки, хорошо параллелится; OOB-оценка даёт бесплатную проверку на данных, не попавших в bootstrap конкретного дерева.


## Boosting

Что если уменьшать не variance, а bias? В таком случае мы используем boosting (бустинг). Изложение дальнейшего материала взято [отсюда](https://education.yandex.ru/handbook/ml/article/gradientnyj-busting)

Каждый следующий базовый алгоритм в бустинге обучается так, чтобы уменьшить общую ошибку всех своих предшественников. Как следствие, итоговая композиция будет иметь меньшее смещение, чем каждый отдельный базовый алгоритм (хотя уменьшение разброса также может происходить):

$$
a(x) = b_1(x) + b_2(x) + ... + b_K(x)
$$



![](boosting.png)

### Интуиция

Давайте представим гольфиста, который хочет попасть мячом в лунку. Гольфист первый раз играет в гольф (так вышло) и он еще не умеет сразу попадать в лунку одним ударом. У гольфиста много попыток, поэтому он с каждой попыткой будет приближаться к лунке, компенсируя свои ошибки в предыдущих попытках:

![](boosting_example.png)

Идея бустинга такая же: давайте компенсировать ошибку предыдущей модели, а потом суммировать результаты. Например, давайте обучим одну какую-то модель на задачу регрессии $b_1(x)$. Предположим, она неидеальна и мы хотим еще лучше. Мы знаем, в каких примерах ошибается модель $b_1(x)$. Например, на объекте $x_1$ модель выдает  предсказание на 10 больше, чем надо:

$$
b_1(x_1) = y_{true} + 10
$$

Что если мы обучим новую модель, которая компенсирует эту ошибку и получим композицию из двух моделей? То есть, обучим модель $b_2(x)$, которая будет предсказывать -10. Тогда сумма результатов этих моделей даст верный ответ:

$$
a(x_1) = b_1(x_1) + b_2(x_1) = (y_{true} + 10) + (-10) = y_{true}
$$

В реальности модель $b_2(x)$ также может ошибаться, поэтому мы обучим модель $b_3(x)$, которая будет компенсировать ошибку композиции предыдущих моделей. Так мы будем продолжать, пока не обучим композицию из $K$ моделей.

Если мы постепенно будем компенсировать ошибки, то с каждой обученой моделью функция потерть будет уменьшаться:

$$
\mathcal{L}(y, a_{i+1}(x)) < \mathcal{L}(y, a_i(x))

### Формализация и сведение к деревьям 

Перед нам стоит задача регрессии, функция потерь $\mathcal{L}(y, x)$. Мы будем использовать композицию из $K$ деревьев:

$$
a(x) = a_K(x) = b_1(x) + b_2(x) + ... + b_K(x)
$$

На первой итерации мы хотим обучить дерево $b_1(x)$ предсказывающее таргет. Мы хотим найти такое дерево из семейства $\mathcal{B}$, которое минимизирует функцию потерь:

$$
b_1(x) = \arg\min_{b \in \mathcal{B}} \mathcal{L}(y, b(x))
$$

Дерево $b_1(x)$ скорее всего делает неточные предсказания и совершает ошибки:

$$
s^1_i = y_i - b_1(x_i)
$$

Пусть дерево $b_2(x)$ эти ошибки компенсирует:

$$
b_2(x) = \arg\min_{b \in \mathcal{B}} \mathcal{L}(s^1, b(x))
$$

А итоговое предсказание:
$$
a_2(x) = b_1(x) + b_2(x)
$$

Если мы захотим обучить дерево $b_3(x)$, то нам надо будет предсказывать ошибки, которые совершает $a_2(x)$ дерево. Давайте обобщим:


Для $k$ дерева "таргеты" будут:

$$
s^{k-1}_i = y_i - a_{k-1}(x_i) = y_i - \sum_{j=1}^{k-1} b_j(x_i)
$$

$$
b_k(x) = \arg\min_{b \in \mathcal{B}} \mathcal{L}(s^{k-1}, b(x))
$$

А композиция:
$$
a_k(x) = a_{k-1}(x) + b_k(x)
$$

### А почему бустинг обычно называют градиентным?

Представим, что наша функция потерь это MSE:

$$
\mathcal{L}(y, x) = \frac{1}{N}\sum_{i=1}^{N}(y_i - a(x_i))^2
$$

Помните как в градиентных методах мы брали производную функции потерь по весам? Давайте посчитаем производную функции потерь по предсказаниям:

$$
\frac{d\mathcal{L}(y_i, z)}{d z} = \frac{d}{d z} (y_i - z)^2 = - 2 (y_i - z) = 2 (z - y_i) = 2 (a_k(x_i) - y_i)
$$

в точке $z = a_k(x_i)$

Разность, на которую обучается $k+1$-ый алгоритм выражается через производную:

$$
s^k_i = y_i - a_{k-1}(x_i) = -2 \frac{d\mathcal{L}(y_i, z)}{d z}
$$

Получается, $k+1$-ая модель учится предсказывать антиградиент функции потерь по предсказанию $k$-ой модели.

Это наблюдение позволяет обобщить подход построения бустинга на **произвольную функцию потерь**. Просто заменим разность $s^k_i$ на антиградиент функции потерь $-g^k_i$:

$$
g^k_i = \frac{d\mathcal{L}(y_i, z)}{d z}
$$

Мы теперь двигаемся не в сторону уменьшения разности между таргетом и предсказанием, а в сторону уменьшения функции потерь. Это позволяет нам работать с ситуациями, когда нет возможности посчитать разность. 

Например, в задаче ранжирования, когда мы предсказываем пару $(i, j)$, где объект $i$ стоит выше чем объект $j$. Здесь у нас нет таргетов и посчитать разность не получится, но дифференцируемая функция потерь есть - благодаря ей мы и будем обучать градиентный бустинг.

### Вернемся к деревьям

При обучении $b_{k+1}$ дерева мы решаем задачу регрессии с таргетом, равным антиградиенту **функции потерь** исходной задачи по предсказаниями $a_k = b_1 + b_2 + ... + b_k$.

Вспоминаем, что при построении дерева мы перебираем спилты и в каждом сплите максимизируем качество:

$$
Q(X, j, t) = H(X_m) - \frac{H(X_l)}{|X_l|} - \frac{H(X_r)}{|X_r|}
$$

При этом минимизируя $H(X)$ - назовем ее **оценочной функцией**. В нашем случае мы ищем антиградиенты **функции потерь**, поэтому в качестве $H(X)$ может выступать среднеквадратичная ошибка или косинусная разность.

В итоге, для обучения $b_{k+1}$ дерева мы:

1) Находим таргеты $h$, то есть антиградиент функции потерь $\mathcal{L}(y_i, z)$:

$$
h_i = -g^k_i = \left.\frac{d \mathcal{L}(y_i, z)}{d z}\right|_{z = a_k(x_i)}
$$

2) Обучаем дерево $b_{k+1}$ на $(x_i, g^k_i)$ как обычно, минимизируя **оценочную функцию**.

#### Темп обучения

Чтобы избежать переобучения, базовые деревья должны быть простыми. Дополнительно мы можем добавить темп обучения:

$$
a_k(x) = a_{k-1}(x) + \eta b_k(x)
$$

### Выводы про бустинг

Поскольку основная цель бустинга — уменьшение смещения, в качестве базовых алгоритмов часто выбирают алгоритмы с высоким смещением и небольшим разбросом. Например, если в качестве базовых классификаторов выступают деревья, то их глубина должна быть небольшой — обычно не больше 2–3 уровней. 

Хотя случайный лес — мощный и достаточно простой для понимания и реализации алгоритм, на практике он чаще всего уступает градиентному бустингу. Поэтому градиентный бустинг сейчас основное продакшн-решение, если работа происходит с табличными данными (в работе с однородными данными — картинками, текстами — доминируют нейросети).

### Наконец-то практика

Есть 3 основные либы, которые используют для обучения. Основное их отличие это метод построения деревьев.

In [ ]:
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

#### XGBoost

Принцип построения дерева: строим дерево последовательно, по уровням, до достижения максимальной глубины. 

В XGBoost деревья стремятся быть симметричными по глубине, и в идеале получается полное бинарное дерево, если это не противоречит другим ограничениям (например, ограничению на минимальное количество объектов в листе). Такие деревья обычно более устойчивы к переобучению.

![](xgboost.png)

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,           # число деревьев в ансамбле
    learning_rate=0.1,          # тем обучения (вклад каждого дерева)
    max_depth=6,                # максимальная глубина дерева
    subsample=0.8,              # доля объектов для каждого дерева
    colsample_bytree=0.8,       # доля признаков для каждого дерева
    random_state=RANDOM_STATE,  # фиксирует воспроизводимость
    n_jobs=-1,                  # использовать все доступные ядра CPU
)

xgb_model.fit(
    X_train_enc, y_train,
    eval_set=[(X_test_enc, y_test)],
    verbose=50
)

res = evaluate(xgb_model, X_train_enc, y_train, X_test_enc, y_test, 'XGBoost (baseline)')
all_results.append(res)

`XGBoost` поддерживает кодирование категориальных фичей, что бывает очень удобно. Нам не нужно самим их обрабатывать (хотя это довольно гибко), достаточно просто передать в класс аргумент `enable_categorical`. В этой библиотеке кодировка OneHot.

In [ ]:
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()

for c in cat_cols:
    X_train_xgb[c] = X_train_xgb[c].astype("category")
    X_test_xgb[c] = X_test_xgb[c].astype("category")


xgb_model = xgb.XGBRegressor(
    enable_categorical=True,
    n_estimators=500,           # число деревьев в ансамбле
    learning_rate=0.1,          # тем обучения (вклад каждого дерева)
    max_depth=6,                # максимальная глубина дерева
    subsample=0.8,              # доля объектов для каждого дерева
    colsample_bytree=0.8,       # доля признаков для каждого дерева
    random_state=RANDOM_STATE,  # фиксирует воспроизводимость
    n_jobs=-1,                  # использовать все доступные ядра CPU
)

xgb_model.fit(
    X_train_xgb, y_train,
    eval_set=[(X_test_xgb, y_test)],
    verbose=50
)

res = evaluate(xgb_model, X_train_xgb, y_train, X_test_xgb, y_test, 'XGBoost (with categorical features)')
all_results.append(res)

#### LightGBM

Принцип построения дерева: на каждом шаге делим вершину с наилучшим скором.

Основным критерием остановки выступает максимально допустимое количество вершин в дереве. Это приводит к тому, что деревья получаются несимметричными, то есть поддеревья могут иметь разную глубину. Например, левое поддерево может иметь глубину 22, а правое может разрастись до глубины 1515.

С одной стороны, это позволяет быстро подогнаться под обучающие данные. С другой — бесконтрольный рост дерева в глубину неизбежно ведёт к переобучению, поэтому LightGBM позволяет, помимо количества вершин, ограничивать и максимальную глубину.

![](lightgbm.png)

In [ ]:
lgb_model = lgb.LGBMRegressor(
    n_estimators=500, 
    learning_rate=0.1, 
    max_depth=6,
    subsample=0.8, 
    colsample_bytree=0.8,
    random_state=RANDOM_STATE, 
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train_enc, y_train,
    eval_set=[(X_test_enc, y_test)],
)
res = evaluate(lgb_model, X_train_enc, y_train, X_test_enc, y_test, 'LightGBM (baseline)')
all_results.append(res)

In [ ]:
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()

for c in cat_cols:
  X_train_lgb[c] = X_train_lgb[c].astype("category")
  X_test_lgb[c] = X_test_lgb[c].astype("category")

lgb_model = lgb.LGBMRegressor(
    n_estimators=500, 
    learning_rate=0.1, 
    max_depth=6,
    subsample=0.8, 
    colsample_bytree=0.8,
    random_state=RANDOM_STATE, 
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train_lgb, y_train,
    eval_set=[(X_test_lgb, y_test)],
    categorical_feature=cat_cols
)
res = evaluate(lgb_model, X_train_lgb, y_train, X_test_lgb, y_test, 'LightGBM (with categorical features)')
all_results.append(res)

#### CatBoost

Принцип построения дерева: все вершины одного уровня имеют одинаковый предикат.

Одинаковые сплиты во всех вершинах одного уровня позволяют избавиться от ветвлений (конструкций if-else) в коде инференса модели с помощью битовых операций и получить более эффективный код, который в разы ускоряет применение модели, особенно в случае применения на батчах.

Кроме этого, такое ограничение на форму дерева выступает в качестве сильной регуляризации, что делает модель более устойчивой к переобучению. Основной критерий остановки, как и в случае XGBoost, — ограничение на глубину дерева. 

In [ ]:
cat_features_idx = X_train.columns.get_indexer(cat_cols)

cb_model = cb.CatBoostRegressor(
    iterations=500, 
    learning_rate=0.1, 
    depth=6,
    subsample=0.8, 
    random_seed=RANDOM_STATE,
    verbose=100,
    cat_features=cat_features_idx
)

cb_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
)

res = evaluate(cb_model, X_train, y_train, X_test, y_test, 'CatBoost (baseline)')
all_results.append(res)

In [ ]:
pd.DataFrame(all_results).sort_values(by="R2_test", ascending=False)

## Stacking


Stacking —  алгоритм ансамблирования, основные отличия которого от предыдущих состоят в следующем:

- Он может использовать алгоритмы разного типа, а не только из какого-то фиксированного семейства. Например, в качестве базовых алгоритмов могут выступать метод ближайших соседей и линейная регрессия.

- Результаты базовых алгоритмов объединяются в один с помощью обучаемой метамодели, а не с помощью какого-либо обычного способа агрегации (суммирования или усреднения).

- Базовые модели обучаются через cross-validation, чтобы не было утечки данных.


Это как собрать команду экспертов (RF, XGBoost, LightGBM), а потом назначить менеджера (Ridge), который решает, кого когда слушать.


### Когда использовать?

- Нужно максимальное качество (последние 0.5–2%)
- Есть разнородные сильные модели с комплементарными ошибками (RF + XGBoost + линейная + …)
- Данных много (десятки тысяч+)
- Готовы к сложному пайплайну и риску переобучения

In [ ]:
estimators = [
    ('rf', RandomForestRegressor(n_estimators=100, max_features='sqrt',
                                 random_state=RANDOM_STATE, n_jobs=-1)),
    ('xgb', xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=RANDOM_STATE, n_jobs=-1)),
    ('lgb', lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=RANDOM_STATE, n_jobs=-1,
                              verbose=-1)),
]

stack = StackingRegressor(
    estimators=estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5, n_jobs=-1
)

stack.fit(X_train_enc, y_train)

res = evaluate(stack, X_train_enc, y_train, X_test_enc, y_test, 'Stacking (RF+XGB+LGB → Ridge)')
all_results.append(res)

In [ ]:
voting = VotingRegressor(
    estimators=estimators, n_jobs=-1
)

voting.fit(X_train_enc, y_train)

res = evaluate(voting, X_train_enc, y_train, X_test_enc, y_test, 'Voting (RF+XGB+LGB, avg)')
all_results.append(res)

## Финальное сравнение

In [ ]:
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('RMSE_test')

styled = results_df.style.format({
    'RMSE_train': '{:,.1f}',
    'RMSE_test': '{:,.1f}',
    'R2_test': '{:.4f}',
}).background_gradient(subset=['R2_test'], cmap='RdYlGn')

styled

## Полезные ссылки

- [XGBoost Documentation](https://xgboost.readthedocs.io/)
- [LightGBM Documentation](https://lightgbm.readthedocs.io/)
- [CatBoost Documentation](https://catboost.ai/docs/)
- [Ансамбли в машинном обучении | Яндекс Учебник](https://education.yandex.ru/handbook/ml/article/ansambli-v-mashinnom-obuchenii)
- [Bias-variance decomposition | Яндекс Учебник](https://education.yandex.ru/handbook/ml/article/bias-variance-decomposition)
- [How to explain gradient boosting](https://explained.ai/gradient-boosting/)
